<a href="https://colab.research.google.com/github/Luke-Dev-Tech/GuardianLens/blob/main/GuardianLens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Food Recognition Model (MasterChef)

# [1] Kaggle

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!mkdir -p modular

mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


# [2] Python utilities Helper Blueprint Methods

## [2.1] Kaggle Helper: Dataset download handling blueprint



In [ ]:
%%writefile modular/kaggle_data.py
import os
import zipfile
from os.path import exists
import kagglehub
from pathlib import Path
import subprocess

# --- Kaggle API Credentials Setup ---
# If you encounter a CalledProcessError during Kaggle download,
# it's likely due to missing or incorrect Kaggle API credentials.
# Follow these steps to set them up:
# 1. Go to your Kaggle account (kaggle.com/me/account).
# 2. In the 'API' section, click 'Create New API Token' to download 'kaggle.json'.
# 3. Upload 'kaggle.json' to your Colab environment (e.g., using the files sidebar). (Legacy API)
# 4. Run the following commands in a separate cell to configure:
#    !mkdir -p ~/.kaggle
#    !mv kaggle.json ~/.kaggle/
#    !chmod 600 ~/.kaggle/kaggle.json
# -----------------------------------

def kaggleDataImport():

  data_path = Path("data")
  image_path = data_path / "food-101"


  #_______________Location Creation_______________

  if not exists(data_path):
      print(f"[Info]: {data_path} doesn't Exists. Creating ... ")
      data_path.mkdir(parents=True, exist_ok=True)
  else:
      print(f"[Success] ==> {data_path} Exists")

  if not exists(image_path):
      # location doesn't Exists means: Data doesn't exists as well
      print(f"[Info]: {image_path} doesn't Exists. Creating ... ")
      image_path.mkdir(parents=True, exist_ok=True)
      #==================Data Writing===================
      # Kaggle doesn't accept raw requests.
      # So we are going to use Kaggle CLI.
      # Kaggle CLI is command line interface command so
      # Subprocess is the way to do that.
      #=================================================
      subprocess.run(
          [
              "kaggle",
              "datasets",
              "download",
              "-d",
              "dansbecker/food-101",
              "-p",
              str(data_path)
          ],
          check=True
      )
      #________________________________________________
      #___________________Unzip File___________________
      filename = "food-101"
      zip_path = data_path / f"{filename}.zip"
      print(f"Unzipping {filename} dataset...")
      with zipfile.ZipFile(zip_path, "r") as zip_ref:
          zip_ref.extractall(image_path)
      #________________________________________________
      #__________________Deleting Zip__________________
      os.remove(zip_path)
      print("RAF-DB ready at:", image_path)
      #________________________________________________
  else:
    print(f"[Success] ===> {image_path} Exists.")


Overwriting modular/kaggle_data.py


## [2.2] Data setup Blueprint Helper
--> **torchvision** - datasets, transformers (datasets.ImageFolder(dir, transformer)

--> **torchvision.utils.data** - DataLoader

--> **collections** - Counter

In [ ]:
%%writefile modular/data_setup.py
import os
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from collections import Counter
import numpy as np
import torch

NUM_WORKERS = os.cpu_count()

def create_dataloaders(
  data_dir: str,
  train_transform: transforms.Compose,
  test_transform: transforms.Compose,
  batch_size: int,
  test_size: float = 0.2,
  random_state: int = 42,
  num_workers: int = NUM_WORKERS,
  apply_weighted_sampler: bool = False
):
    train_data = datasets.ImageFolder(root=data_dir, transform=train_transform)
    test_data = datasets.ImageFolder(root=data_dir, transform=test_transform)

    class_names = train_data.classes
    indices = list(range(len(train_data)))

    train_idx, test_idx = train_test_split(
        indices,
        test_size=test_size,
        random_state=random_state,
        shuffle=True,
        stratify=train_data.targets
    )

    train_dataset = Subset(train_data, train_idx)
    test_dataset = Subset(test_data, test_idx)

    print(f"[INFO] Total images: {len(train_data)}")
    print(f"[INFO] Training images: {len(train_dataset)}")
    print(f"[INFO] Testing images: {len(test_dataset)}")


    sampler = None
    if apply_weighted_sampler:
        print("[INFO] Applying WeightedRandomSampler for class imbalance...")
        train_labels = np.array(train_data.targets)[train_idx]

        class_counts = np.bincount(train_labels)
        class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
        sample_weights = class_weights[train_labels]

        sampler = torch.utils.data.WeightedRandomSampler(
            sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        train_dataloader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=True,
            sampler=sampler
        )
    else:
        train_dataloader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=True
        )

    test_dataloader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return (train_dataloader, test_dataloader, class_names, train_dataset, test_dataset, train_idx, test_idx)

Overwriting modular/data_setup.py


## [2.3] Engine Blueprint: Training and Testing



In [ ]:
%%writefile modular/engine.py

import torch
from tqdm.auto import tqdm
from typing import Dict, List, Optional


def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fun: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device):
    model.train()
    train_loss, train_acc = 0, 0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fun(y_pred, y)
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)

    return train_loss / len(dataloader), train_acc / len(dataloader)


def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fun: torch.nn.Module,
              device: torch.device):       # optimizer removed — test step never needs it
    model.eval()
    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            loss = loss_fun(y_pred, y)
            test_loss += loss.item()
            test_acc += (torch.argmax(torch.softmax(y_pred, dim=1), dim=1) == y).sum().item() / len(y_pred)

    return test_loss / len(dataloader), test_acc / len(dataloader)


def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fun: torch.nn.Module,
          epochs: int,
          device: torch.device,
          scheduler=None,          # NEW: pass your CosineAnnealingLR here
          early_stopping=None      # NEW: pass your EarlyStopping instance here
          ) -> Dict[str, List]:

    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model, train_dataloader, loss_fun, optimizer, device)
        test_loss, test_acc = test_step(model, test_dataloader, loss_fun, device)

        # ---- Scheduler step (once per epoch, after optimiser step) ----
        current_lr = optimizer.param_groups[0]['lr']
        if scheduler is not None:
            scheduler.step()

        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f} | "
            f"lr: {current_lr:.6f}"
        )

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        # ---- Early stopping check (after printing so you see the epoch) ----
        if early_stopping is not None:
            early_stopping(test_loss, model)
            if early_stopping.stop:
                print(f"\n[EarlyStopping] Triggered at epoch {epoch+1}. Best val_loss: {early_stopping.best_loss:.4f}")
                early_stopping.restore_best(model)   # reload the best weights
                break

    return results

Overwriting modular/engine.py


## [2.3a] Engine Timer Helper: Timing the Training and Testing Process

In [ ]:
%%writefile modular/engine_with_time_count.py
import torch
from timeit import default_timer as timer
def engine_with_time_count(model, train_dataloader, test_dataloader,
                            loss_fun, optimizer, epoch_num, device,
                            scheduler=None, early_stopping=None):
    torch.manual_seed(42)
    torch.cuda.manual_seed(42)
    start_time = timer()
    engine.train(
        model=model,
        train_dataloader=train_dataloader,
        test_dataloader=test_dataloader,
        loss_fun=loss_fun,
        optimizer=optimizer,
        epochs=epoch_num,
        device=device,
        scheduler=scheduler,
        early_stopping=early_stopping
    )
    end_time = timer()
    print(f"[INFO] Total training time: {end_time - start_time:.3f} seconds")

Overwriting modular/engine_with_time_count.py


## [2.4] Save Model Blueprint: Saving the State

In [ ]:
%%writefile modular/utils.py

import torch
from pathlib import Path

def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name:str):
  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True, exist_ok=True)

  model_save_path = target_dir_path / model_name

  print(f"[INFO] Saving model to: {model_save_path}")
  torch.save(obj=model.state_dict(), f=model_save_path)

Overwriting modular/utils.py


## [2.5] Pred And Plot individual image Blueprint

In [ ]:
%%writefile modular/pred_and_plot_img.py
from typing import List, Tuple
import torch
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'


def pred_and_plot_img(
    model: torch.nn.Module,
    image_path: str,
    class_name: List[str],
    image_size: Tuple[int, int] = (224, 224),
    transform: torchvision.transforms = None,
    device: torch.device = None
):
    # Load image
    img_pil = Image.open(image_path).convert("RGB")

    # Use provided transform or default VGG-Face transform
    if transform is None:
        transform = torchvision.transforms.Compose([
            torchvision.transforms.Resize(image_size),
            torchvision.transforms.Grayscale(num_output_channels=3),
            torchvision.transforms.ToTensor(),
            transforms.Normalize(
              mean=[0.485, 0.456, 0.406],
              std=[0.229, 0.224, 0.225]
          )
        ])

    model.to(device)
    model.eval()

    with torch.inference_mode():
        img_tensor = transform(img_pil).unsqueeze(0).to(device)
        logits = model(img_tensor)
        probs = torch.softmax(logits, dim=1)
        pred_idx = probs.argmax(dim=1).item()
        pred_prob = probs.max().item()

    actual_label = Path(image_path).parent.name
    pred_label = class_name[pred_idx]

    plt.figure(figsize=(5, 5))
    plt.imshow(img_pil)
    plt.title(
        f"Actual: {actual_label} | Pred: {pred_label} | Prob: {pred_prob:.3f}"
    )
    plt.axis("off")


Overwriting modular/pred_and_plot_img.py


## [2.6] Early Stopping Helper

In [ ]:
# ============================================================
# Early Stopping
# ============================================================
import torch
import numpy as np
import copy

class EarlyStopping:
    """
    Monitors val_loss each epoch.
    Saves best model weights when val_loss improves.
    Stops training if val_loss hasn't improved for `patience` epochs.
    Restores best weights at the end.

    Parameters
    ----------
    patience : int
        How many epochs to wait after last improvement before stopping.
    min_delta : float
        Minimum change in val_loss to count as an improvement.
        Prevents stopping on tiny, noise-level improvements.
    verbose : bool
        Whether to print messages when saving or about to stop.
    """

    def __init__(self, patience: int = 5, min_delta: float = 1e-4, verbose: bool = True):
        self.patience   = patience
        self.min_delta  = min_delta
        self.verbose    = verbose

        self.best_loss  = np.inf      # track the lowest val_loss seen
        self.counter    = 0           # epochs without improvement
        self.stop       = False       # flag checked by train loop
        self.best_state = None        # saved model weights at best epoch

    def __call__(self, val_loss: float, model: torch.nn.Module):
        """Called once per epoch with the current val_loss and model."""

        if val_loss < self.best_loss - self.min_delta:
            # Improvement found — save the weights and reset counter
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_state = copy.deepcopy(model.state_dict())
            if self.verbose:
                print(f"  [EarlyStopping] ✓ val_loss improved to {val_loss:.4f} — model saved")
        else:
            # No improvement
            self.counter += 1
            if self.verbose:
                print(f"  [EarlyStopping] No improvement ({self.counter}/{self.patience})")
            if self.counter >= self.patience:
                self.stop = True

    def restore_best(self, model: torch.nn.Module):
        """Load the best saved weights back into the model."""
        if self.best_state is not None:
            model.load_state_dict(self.best_state)
            print(f"  [EarlyStopping] Best weights restored (val_loss: {self.best_loss:.4f})")

# [3] Pipeline

In [ ]:
import os
import torch
from modular import kaggle_data, data_setup, engine, engine_with_time_count, utils, pred_and_plot_img
import torchvision.transforms as transforms
import torchvision
import importlib
from collections import Counter
from torch.utils.data import DataLoader, Subset
import timm
from timm import data
#=================IMPORT KAGGLE DATA=========================
kaggle_data.kaggleDataImport()
#====================Device Setup=============================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
#=============================================================


[Success] ==> data Exists
[Success] ===> data/food-101 Exists.


## [3.0]. Construction: Model

In [ ]:
import timm.data
import numpy as np

from urllib.request import urlopen
from PIL import Image
import timm

model = timm.create_model(
    'convnext_base.fb_in1k',
    pretrained=True,
    num_classes=101,
    in_chans=3
)
model = model.eval()

## [3.1] Construction: Transformers

In [ ]:
data_config = timm.data.resolve_model_data_config(model)
train_transforms = timm.data.create_transform(**data_config, is_training=True)   # includes augmentation
test_transforms  = timm.data.create_transform(**data_config, is_training=False)  # just resize + normalize

## [3.2] Construction: DataLoaders

In [ ]:
#=================Attribute Section===========================
# NUM_EPOCHS = 30
BATCH_SIZE = 32
# HIDDEN_UNITS = 10
# LEARNING_RATE = 0.0005
data_dir = "data/food-101/food-101/food-101/images"
#=============================================================

train_dataloader, test_dataloader, class_names , train_dataset, test_dataset, train_index, test_index = data_setup.create_dataloaders(
  data_dir=data_dir,
  train_transform=train_transforms,
  test_transform=test_transforms,
  batch_size=BATCH_SIZE,
  test_size=0.2,  # 80/20 split
  apply_weighted_sampler=True
)
#=============================================================

[INFO] Total images: 101000
[INFO] Training images: 80800
[INFO] Testing images: 20200
[INFO] Applying WeightedRandomSampler for class imbalance...


## [3.3] Construction: Loss Function

In [ ]:
from collections import Counter
import torch

subset = train_dataloader.dataset
train_targets = [subset.dataset.targets[i] for i in subset.indices]

counts = Counter(train_targets)
num_classes = len(class_names)
total_samples = sum(counts.values())

class_weights = torch.tensor(
    [total_samples / (num_classes * counts.get(i, 1)) for i in range(num_classes)],
    dtype=torch.float
).to(device)

print("Class weights:")
for name, w in zip(class_names, class_weights):
    print(f"  {name}: {w:.3f}")

loss_fun = torch.nn.CrossEntropyLoss(weight=class_weights)

Class weights:
  apple_pie: 1.000
  baby_back_ribs: 1.000
  baklava: 1.000
  beef_carpaccio: 1.000
  beef_tartare: 1.000
  beet_salad: 1.000
  beignets: 1.000
  bibimbap: 1.000
  bread_pudding: 1.000
  breakfast_burrito: 1.000
  bruschetta: 1.000
  caesar_salad: 1.000
  cannoli: 1.000
  caprese_salad: 1.000
  carrot_cake: 1.000
  ceviche: 1.000
  cheese_plate: 1.000
  cheesecake: 1.000
  chicken_curry: 1.000
  chicken_quesadilla: 1.000
  chicken_wings: 1.000
  chocolate_cake: 1.000
  chocolate_mousse: 1.000
  churros: 1.000
  clam_chowder: 1.000
  club_sandwich: 1.000
  crab_cakes: 1.000
  creme_brulee: 1.000
  croque_madame: 1.000
  cup_cakes: 1.000
  deviled_eggs: 1.000
  donuts: 1.000
  dumplings: 1.000
  edamame: 1.000
  eggs_benedict: 1.000
  escargots: 1.000
  falafel: 1.000
  filet_mignon: 1.000
  fish_and_chips: 1.000
  foie_gras: 1.000
  french_fries: 1.000
  french_onion_soup: 1.000
  french_toast: 1.000
  fried_calamari: 1.000
  fried_rice: 1.000
  frozen_yogurt: 1.000
  gar

## [3.4] Transfer learning: Fine Tuning Phases
--> Primary Focuses:

    1. Freezing | Unfreezing Heads of loaded Model
    2. Optimizer Adjustment (Learning rates | weight decays | gradient)

In [ ]:
print(model)                          # full structure
# or just the top-level module names:
print([name for name, _ in model.named_children()])

ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((128,), eps=1e-06, elementwise_affine=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
          (norm): LayerNorm((128,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=128, out_features=512, bias=True)
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Linear(in_features=512, out_features=128, bias=True)
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), g

### Phase [1]

In [ ]:
#=========================={PHASE 1}===============================#
# PHASE (1): Backbone
# EPOCH - 5
# LR - 0.001
# Backbone
# New optimizer [Mandatory]
# Weight decay [ADDed]
#=============================================================
# Backbone phase: Freze all the hidden layers
for param in model.features.parameters():
    param.requires_grad = False

#====================Phase 1 Optimizer and scheduler========================
optimizer_p1 = torch.optim.Adam(model.classifier.parameters(), lr=0.001)

# Scheduler
scheduler_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p1,
    T_max=15,
    eta_min=1e-6
)

# Early Stopping - From EarlyStopping Class
early_stopping_p1 = EarlyStopping(patience=5, min_delta=1e-4, verbose=True)
#=============================================================
print("======== PHASE 1: Training ==========")
engine_with_time_count(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fun=loss_fun,
    optimizer=optimizer_p1,
    epoch_num=15,           # max epochs — may stop earlier
    device=device,
    scheduler=scheduler_p1,
    early_stopping=early_stopping_p1
)
print("=======================================")
#=============================================================

AttributeError: 'ConvNeXt' object has no attribute 'features'

### Phase [2]

In [ ]:
# #======================={PHASE 2}============================#
# # PHASE (2): Fine-Tuning (last 1 block )
# # EPOCH - 12
# # LR - 0.0001
# # Unfreeze last block
# # New optimizer [Mandatory]
# # Weight decay [ADDed]
# # Phase 2: Model slightly adapts high-level facial features (hopefully)
# #=============================================================
# # Fine Tuning Level: Unfreeze one layer
#======================={PHASE 2}============================#
# Unfreeze last backbone block for fine-tuning
for param in model.features[-1:].parameters():
    param.requires_grad = True

optimizer_p2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001,         # 10x lower than Phase 1 — protects pretrained weights
    weight_decay=1e-4
)

scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p2, T_max=10, eta_min=1e-7
)

# Fresh early stopping instance for Phase 2
early_stopping_p2 = EarlyStopping(patience=5, min_delta=1e-4, verbose=True)

print("======== PHASE 2: Fine-Tuning Last Block ==========")
engine_with_time_count(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fun=loss_fun,
    optimizer=optimizer_p2,
    epoch_num=10,
    device=device,
    scheduler=scheduler_p2,
    early_stopping=early_stopping_p2
)
print("====================================================")

### Phase [3]

In [ ]:
#======================={PHASE 3}============================#
# Unfreeze last 3 blocks instead of just 1
for param in model.features[-3:].parameters():
    param.requires_grad = True

optimizer_p3 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.00001,        # even lower — more blocks unfrozen = more risk of destroying weights
    weight_decay=1e-4
)

scheduler_p3 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p3, T_max=15, eta_min=1e-8
)

early_stopping_p3 = EarlyStopping(patience=5, min_delta=1e-4, verbose=True)

print("======== PHASE 3: Deeper Fine-Tuning ==========")
engine_with_time_count(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fun=loss_fun,
    optimizer=optimizer_p3,
    epoch_num=15,
    device=device,
    scheduler=scheduler_p3,
    early_stopping=early_stopping_p3
)
print("================================================")

### Phase [4]

In [ ]:
#======================={PHASE 4}============================#
# Unfreeze last 5 blocks — going deeper into the backbone
for param in model.features[-5:].parameters():
    param.requires_grad = True

optimizer_p4 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.000005,       # lower than Phase 3 — more blocks = more risk
    weight_decay=1e-4
)

scheduler_p4 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p4, T_max=15, eta_min=1e-8
)

early_stopping_p4 = EarlyStopping(patience=5, min_delta=1e-4, verbose=True)

print("======== PHASE 4: Deeper Fine-Tuning ==========")
engine_with_time_count(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fun=loss_fun,
    optimizer=optimizer_p4,
    epoch_num=15,
    device=device,
    scheduler=scheduler_p4,
    early_stopping=early_stopping_p4
)
print("================================================")

### Phase [5]

In [ ]:
#======================={PHASE 5}============================#
# Unfreeze everything — full model fine-tuning
for param in model.parameters():
    param.requires_grad = True

optimizer_p5 = torch.optim.Adam(
    model.parameters(),
    lr=0.000001,       # very low — entire backbone is exposed
    weight_decay=1e-4
)

scheduler_p5 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p5, T_max=15, eta_min=1e-9
)

early_stopping_p5 = EarlyStopping(patience=5, min_delta=1e-4, verbose=True)

print("======== PHASE 5: Full Model Fine-Tuning ==========")
engine_with_time_count(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fun=loss_fun,
    optimizer=optimizer_p5,
    epoch_num=20,
    device=device,
    scheduler=scheduler_p5,
    early_stopping=early_stopping_p5
)
print("==================================================")

## Saving Model

In [ ]:
#=================Utils for saving models=====================
utils.save_model(
    model=model,
    target_dir='models',
    model_name=f'FaceEmotionDection_argumented_final.pth'
)
#=============================================================

# [5] Random 3 IMG Testing

In [ ]:
from typing import List, Tuple
import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image # Import Image for PIL operations
from pathlib import Path # Import Path to handle paths


def pred_and_plot_img(model: torch.nn.Module,
                      image_path:str,
                      class_name: List[str],
                      image_size: Tuple[int, int] = (224, 224),
                      transform: torchvision.transforms = None,
                      device: torch.device = device):
  # Load image as PIL Image first
  img_pil = Image.open(image_path).convert("RGB") # Ensure it's RGB

  if transform is not None:
    img_transform = transform
  else:
    img_transform = torchvision.transforms.Compose([
        torchvision.transforms.Resize(image_size),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
  model.to(device)
  model.eval()
  with torch.inference_mode():
    # Apply transform to the PIL image
    transformed_img = img_transform(img_pil).unsqueeze(dim=0) # this is adding a batch size (1 img)
    target_image_pred = model(transformed_img.to(device))
    target_image_pred_probs = torch.softmax(target_image_pred, dim=1) # sigmoid
    target_image_pred_label = torch.argmax(target_image_pred_probs, dim=1) # return index

    # Extract actual label from image_path
    actual_label = Path(image_path).parent.name

    plt.figure()
    # To plot, convert the PIL image to a tensor (without normalization, just to display original appearance)
    img_tensor_for_plot = torchvision.transforms.ToTensor()(img_pil)
    plt.imshow(img_tensor_for_plot.permute(1, 2, 0)) # Permute to HWC for matplotlib
    plt.title(f"Actual: {actual_label} | Pred: {class_name[target_image_pred_label]} | Prob: {target_image_pred_probs.max():.3f}")
    plt.axis(False)



import random
num_images_to_plot = 3
test_image_paths = list(Path(test_dir).glob("*/*.jpg"))
random_image_paths = random.sample(test_image_paths, k=num_images_to_plot)

for image_path in random_image_paths:
  pred_and_plot_img(
      model=model,
      image_path=image_path,
      class_name=class_names,
      transform=test_transform,
      device=device
  )

# [6] Custom IMG TESTing

In [ ]:
import requests
from pathlib import Path
import random

def custom_img_testing(link: str):
  custom_img_path = Path(f'data/custom_img_{round(random.random(), 3)}.jpg')

  if not custom_img_path.is_file():
    print(f"Downloading custom image ... ")
    with open(custom_img_path, "wb") as f:
      request = requests.get(link)
      f.write(request.content)
  else:
    print(f"{custom_img_path} already exists")

  return custom_img_path


pred_and_plot_img(
    model=model,
    image_path=custom_img_testing(link="https://media.allure.com/photos/657789b375d47454b054df1f/16:9/w_2560%2Cc_limit/sydney%2520sweeney%252023.jpg"),
    class_name=class_names,
    transform=test_transform,
    device=device
)


In [ ]:
y_true = []
y_pred = []

model.eval()
with torch.inference_mode():
  for X, y in test_dataloader:
    X, y = X.to(device), y.to(device)

    # Forward pass
    logits = model(X)

    # Convert logits to predicted labels
    predicted_labels = torch.argmax(torch.softmax(logits, dim=1), dim=1)

    # Store true and predicted labels
    y_true.extend(y.cpu().numpy())
    y_pred.extend(predicted_labels.cpu().numpy())

print("Predictions generated successfully.")
print(f"Number of true labels collected: {len(y_true)}")
print(f"Number of predicted labels collected: {len(y_pred)}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Compute the confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Display the confusion matrix
plt.figure(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right') # Rotate labels for better readability
plt.tight_layout()
plt.show()

print("Confusion matrix displayed successfully.")

In [ ]:
from sklearn.metrics import classification_report

# Generate a classification report
report = classification_report(y_true, y_pred, target_names=class_names)

print("Classification Report:")
print(report)
